# Stage 5 — SHAP Interpretability Analysis

**Thesis:** Predictive Analytics for MSME Credit Risk Assessment using Behavioural Feature Engineering and Explainable Ensemble Machine Learning

Notebook 5 of 5, the last one. I'm applying SHAP (Lundberg & Lee, 2017) to the
best model from Notebook 4 (XGBoost) to try and answer:

- **RQ1** - which behavioural features are actually most predictive of MSME default?
- **RQ4** - what do SHAP values tell me about individual decisions, and how does
  that translate into something a lender could actually use?

What's in here:

1. Load the tuned XGBoost model and the held-out test partition
2. `TreeExplainer` on a **stratified 5,000-instance test sample** (per the research design)
3. Global feature importance - mean |SHAP| bar plot
4. Beeswarm summary - direction and magnitude of each feature's effect
5. Individual **waterfall** charts for a representative high-risk and low-risk borrower
6. A SHAP **stability check** across random seeds (this addresses risk R4 from my proposal)
7. Turning all of this into lender-facing guidance

Outputs: figures `shap_01`–`shap_05`, `outputs/shap_feature_importance.csv`.


In [1]:

import os, warnings, joblib, re
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

warnings.filterwarnings("ignore")
plt.ioff()
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
OUT_DIR = os.path.join(ROOT, "outputs")
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
                     "font.family": "DejaVu Sans", "font.size": 11})

def savefig(fig, name, caption=""):
    fig.savefig(os.path.join(OUT_DIR, name), dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved -> outputs/{name}" + (f"   |  {caption}" if caption else ""))

# load the model, its preprocessor, and the test partition
bm = joblib.load(os.path.join(OUT_DIR, "best_model.joblib"))
model, preprocessor = bm["classifier"], bm["preprocessor"]
feat_names_raw = bm["feature_names"]
num_cols, cat_cols = bm["num_cols"], bm["cat_cols"]
print("best model:", bm["model_name"], bm["best_params"])

test = pd.read_parquet(os.path.join(OUT_DIR, "split_test.parquet")).set_index("SK_ID_CURR")
y_test = test.pop("TARGET").astype(int)
X_test = test[num_cols + cat_cols]
Xte = pd.DataFrame(preprocessor.transform(X_test), columns=feat_names_raw, index=X_test.index)
print("test matrix:", Xte.shape, " default rate", round(y_test.mean(), 3))


C:\Users\VARUN\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


best model: XGBoost {'learning_rate': 0.05, 'max_depth': 3}


test matrix: (7683, 104)  default rate 0.102


In [2]:
# human-readable labels for the plots - the raw column names are useful for code
# but unreadable in a figure, so map them to something a reader can follow
PRETTY = {
    "repay_consistency": "Repay", "payment_shortfall": "Shortfall",
    "delinquency": "Delinq", "credit_utilisation": "CardUtil",
    "bureau_depth": "Bureau", "prev_app": "PrevApp", "raw": "", "has": "Has",
}
MANUAL = {
    "raw__EXT_SOURCE_1": "External credit score 1",
    "raw__EXT_SOURCE_2": "External credit score 2",
    "raw__EXT_SOURCE_3": "External credit score 3",
    "raw__DAYS_BIRTH": "Age (days, negative)",
    "raw__DAYS_EMPLOYED": "Employment length (days, negative)",
    "raw__DAYS_ID_PUBLISH": "Days since ID document issued",
    "raw__DAYS_LAST_PHONE_CHANGE": "Days since last phone change",
    "raw__AMT_CREDIT": "Loan amount", "raw__AMT_ANNUITY": "Loan annuity",
    "raw__AMT_INCOME_TOTAL": "Annual income", "raw__AMT_GOODS_PRICE": "Goods price",
    "raw__credit_income_ratio": "Loan-to-income ratio",
    "raw__annuity_income_ratio": "Annuity-to-income ratio",
    "raw__credit_goods_ratio": "Loan-to-goods-price ratio",
    "raw__credit_term": "Implied loan term (months)",
    "raw__REGION_RATING_CLIENT": "Region rating",
    "raw__CODE_GENDER_F": "Gender: female", "raw__CODE_GENDER_M": "Gender: male",
    "raw__NAME_EDUCATION_TYPE_Higher education": "Education: higher",
    "raw__NAME_FAMILY_STATUS_Married": "Family status: married",
    "raw__NAME_CONTRACT_TYPE_Cash loans": "Contract type: cash loan",
    "raw__FLAG_OWN_CAR_Y": "Owns a car",
    "repay_consistency__n_instalments": "Number of past instalments",
    "repay_consistency__ontime_ratio": "On-time instalment ratio",
    "repay_consistency__late_ratio": "Late instalment ratio",
    "repay_consistency__days_late_mean": "Mean days late (instalments)",
    "repay_consistency__days_late_max": "Max days late (instalments)",
    "repay_consistency__dpd30_ratio": "Share of instalments 30+ DPD",
    "payment_shortfall__short_ratio": "Share of underpaid instalments",
    "payment_shortfall__pay_ratio_min": "Worst payment-to-due ratio",
    "payment_shortfall__pay_ratio_mean": "Mean payment-to-due ratio",
    "payment_shortfall__shortfall_ratio_mean": "Mean instalment shortfall",
    "delinquency__months_since_last_dpd": "Months since last delinquency",
    "delinquency__dpd_months_count": "Number of delinquent months",
    "delinquency__dpd_month_ratio": "Share of delinquent months",
    "delinquency__dpd_max": "Worst days-past-due",
    "delinquency__bureau_day_overdue_max": "Bureau: max days overdue",
    "credit_utilisation__util_mean": "Mean credit-card utilisation",
    "credit_utilisation__util_max": "Peak credit-card utilisation",
    "credit_utilisation__util_recent": "Recent credit-card utilisation",
    "credit_utilisation__util_slope": "Credit-card utilisation trend",
    "credit_utilisation__drawings_mean": "Mean credit-card drawings",
    "bureau_depth__n_bureau_lines": "Number of bureau credit lines",
    "bureau_depth__n_active_lines": "Active bureau credit lines",
    "bureau_depth__debt_credit_ratio": "Bureau debt-to-credit ratio",
    "bureau_depth__credit_sum_total": "Total bureau credit",
    "bureau_depth__history_days": "Length of credit history (days)",
    "bureau_depth__active_line_ratio": "Share of active credit lines",
    "bureau_depth__bb_pastdue_ratio": "Bureau: share of past-due months",
    "bureau_depth__bb_months_observed": "Bureau: months of history",
    "prev_app__approval_rate": "Prior-application approval rate",
    "prev_app__refusal_rate": "Prior-application refusal rate",
    "prev_app__grant_ratio_mean": "Granted-to-requested amount ratio",
    "prev_app__amt_application_mean": "Mean prior-application amount",
    "prev_app__amt_credit_mean": "Mean prior-application credit",
    "prev_app__n_prev_apps": "Number of prior applications",
    "prev_app__days_since_last_app": "Days since last application",
}
def pretty(col):
    if col in MANUAL:
        return MANUAL[col]
    fam, _, rest = col.partition("__")
    if fam == "has" and rest:
        return "Has " + rest.replace("_", " ") + " history"
    if fam in PRETTY and rest:
        tag = PRETTY[fam]
        rest = rest.replace("_", " ")
        return (f"{tag}: {rest}" if tag else rest).strip().capitalize()
    return re.sub(r"^raw__", "", col).replace("_", " ")

feat_names = [pretty(c) for c in feat_names_raw]
Xte.columns = feat_names
print("example labels:", feat_names[:6])


example labels: ['Number of past instalments', 'On-time instalment ratio', 'Late instalment ratio', 'Share of instalments 30+ DPD', 'Mean days late (instalments)', 'Max days late (instalments)']


## 1. TreeExplainer on a stratified 5,000-instance test sample

`TreeExplainer` computes exact SHAP values for tree ensembles in polynomial time
(Lundberg & Lee, 2017), which is what makes this feasible at all. Per the research
design, I'm running it on a stratified 5,000-row sample of the held-out test
partition rather than the whole thing. SHAP values sit on the model's log-odds
(margin) scale - positive pushes the prediction toward **default**.


In [3]:
from sklearn.model_selection import train_test_split

# stratified 5,000-row sample of the test partition
idx = np.arange(len(Xte))
samp_idx, _ = train_test_split(idx, train_size=5000, stratify=y_test.values,
                               random_state=RANDOM_STATE)
X_samp = Xte.iloc[samp_idx]
y_samp = y_test.values[samp_idx]
print(f"SHAP sample: {len(X_samp):,} rows, {y_samp.mean()*100:.1f}% default")

explainer = shap.TreeExplainer(model)
sv = explainer(X_samp)                       # Explanation object, margin scale
print("SHAP values shape:", sv.values.shape, " base value:", round(float(np.ravel(sv.base_values)[0]), 3))

mean_abs = np.abs(sv.values).mean(axis=0)
imp = (pd.Series(mean_abs, index=feat_names)
         .sort_values(ascending=False)
         .rename("mean_abs_shap"))
imp.to_csv(os.path.join(OUT_DIR, "shap_feature_importance.csv"))
print("\nTop 15 features by mean |SHAP|:")
print(imp.head(15).round(4).to_string())


SHAP sample: 5,000 rows, 10.2% default


SHAP values shape: (5000, 104)  base value: -0.0

Top 15 features by mean |SHAP|:
External credit score 2               0.3777
External credit score 3               0.2340
External credit score 1               0.1844
Loan-to-goods-price ratio             0.1361
Granted-to-requested amount ratio     0.1246
Employment length (days, negative)    0.1143
Late instalment ratio                 0.1072
Implied loan term (months)            0.0933
Gender: female                        0.0916
Number of past instalments            0.0857
Loan annuity                          0.0853
Recent credit-card utilisation        0.0706
Education: higher                     0.0662
Mean prior-application amount         0.0532
Prior-application refusal rate        0.0518


## 2. Global feature importance (mean |SHAP|)

Features ordered by mean absolute SHAP value across the 5,000-instance sample.
I've coloured the bars by source so it's obvious at a glance which ones are
**behavioural** (engineered from the sub-tables), **raw** (straight from the main
application table), and **coverage** (the `has_*` indicators).


In [4]:
import seaborn as sns
sns.set_style("whitegrid")

def source_of(raw_col):
    fam = raw_col.split("__")[0]
    if fam == "raw":  return "raw"
    if fam == "has":  return "coverage"
    return "behavioural"

src = pd.Series([source_of(c) for c in feat_names_raw], index=feat_names)
SRC_COLOR = {"behavioural": "#4C72B0", "raw": "#B0B7C0", "coverage": "#8C6BB1"}

top = imp.head(20)[::-1]
colors = [SRC_COLOR[src[f]] for f in top.index]

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top.index, top.values, color=colors, edgecolor="black", linewidth=0.4)
ax.set_xlabel("Mean |SHAP value|  (impact on model output, log-odds)")
ax.set_title("Global feature importance — XGBoost (SHAP)", fontweight="bold")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in SRC_COLOR.values()]
ax.legend(handles, SRC_COLOR.keys(), loc="lower right", frameon=True)
sns.despine(ax=ax)
savefig(fig, "shap_01_global_importance.png",
        "Top 20 features by mean absolute SHAP value for the XGBoost model on a "
        "stratified 5,000-instance test sample, coloured by feature source.")

n_beh = (src.reindex(imp.head(20).index) == "behavioural").sum()
print(f"{n_beh} of the top-20 features are engineered behavioural features")


saved -> outputs/shap_01_global_importance.png   |  Top 20 features by mean absolute SHAP value for the XGBoost model on a stratified 5,000-instance test sample, coloured by feature source.
9 of the top-20 features are engineered behavioural features


## 3. Beeswarm summary — direction and magnitude of effects

Each dot is one borrower. Horizontal position is that feature's SHAP value for
that borrower (further right = pushes harder toward default), and colour is the
feature's actual value (red = high, blue = low).


In [5]:
plt.figure()
shap.plots.beeswarm(sv, max_display=18, show=False, color_bar=True)
fig = plt.gcf()
fig.set_size_inches(10, 8)
fig.suptitle("SHAP beeswarm summary — XGBoost (MSME proxy)", y=1.01, fontweight="bold")
savefig(fig, "shap_02_beeswarm.png",
        "SHAP beeswarm summary for the XGBoost model on the 5,000-instance test "
        "sample. Each point is one borrower; rightward positions push the prediction "
        "toward default, and colour encodes the feature value.")


saved -> outputs/shap_02_beeswarm.png   |  SHAP beeswarm summary for the XGBoost model on the 5,000-instance test sample. Each point is one borrower; rightward positions push the prediction toward default, and colour encodes the feature value.


## 4. Behavioural features only — answering RQ1

Now I'll restrict the SHAP ranking to just the engineered behavioural features,
setting the external scores and raw application fields aside, to isolate which
*behavioural* signals the model is actually leaning on most.


In [6]:
beh_mask = src == "behavioural"
beh_imp = imp[imp.index.isin(src[beh_mask].index)].head(15)[::-1]

# colour by which family each behavioural feature came from
fam_lookup = {pretty(c): c.split("__")[0] for c in feat_names_raw}
FAM_COLOR = {"repay_consistency": "#4C72B0", "payment_shortfall": "#DD8452",
             "delinquency": "#C44E52", "credit_utilisation": "#8C6BB1",
             "bureau_depth": "#5B8C7B", "prev_app": "#937860"}
bcolors = [FAM_COLOR.get(fam_lookup.get(f), "#999") for f in beh_imp.index]

fig, ax = plt.subplots(figsize=(8, 6.5))
ax.barh(beh_imp.index, beh_imp.values, color=bcolors, edgecolor="black", linewidth=0.4)
ax.set_xlabel("Mean |SHAP value|")
ax.set_title("Most predictive behavioural features (SHAP) — RQ1", fontweight="bold")
handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAM_COLOR.values()]
ax.legend(handles, [k.replace("_", " ") for k in FAM_COLOR], loc="lower right", fontsize=8)
sns.despine(ax=ax)
savefig(fig, "shap_03_behavioural_importance.png",
        "The 15 most influential engineered behavioural features by mean absolute "
        "SHAP value, coloured by feature family. This is the SHAP-based answer to RQ1.")

print("Top behavioural drivers of default (SHAP):")
print(beh_imp[::-1].round(4).to_string())


saved -> outputs/shap_03_behavioural_importance.png   |  The 15 most influential engineered behavioural features by mean absolute SHAP value, coloured by feature family. This is the SHAP-based answer to RQ1.
Top behavioural drivers of default (SHAP):
Granted-to-requested amount ratio    0.1246
Late instalment ratio                0.1072
Number of past instalments           0.0857
Recent credit-card utilisation       0.0706
Mean prior-application amount        0.0532
Prior-application refusal rate       0.0518
Total bureau credit                  0.0371
Bureau debt-to-credit ratio          0.0361
Length of credit history (days)      0.0357
Repay: days late std                 0.0347
Days since last application          0.0340
Peak credit-card utilisation         0.0339
Prior-application approval rate      0.0293
Active bureau credit lines           0.0235
Share of underpaid instalments       0.0234


## 5. Individual explanations (waterfall charts)

Waterfall charts break down a single prediction: starting from the sample's base
value (the average log-odds), each feature nudges the prediction up (toward
default) or down (toward repaid). I'm showing two contrasting, correctly
classified borrowers - one the model scores as high-risk, one as low-risk.


In [7]:
proba_samp = model.predict_proba(X_samp.values)[:, 1]

# high-risk: actually defaulted, and the model was confident about it
hr = np.where((y_samp == 1) & (proba_samp > np.quantile(proba_samp, 0.97)))[0][0]
# low-risk: actually repaid, and the model was confident about that too
lr = np.where((y_samp == 0) & (proba_samp < np.quantile(proba_samp, 0.03)))[0][0]

for i, tag, cap in [(hr, "highrisk", "high-risk borrower (actual: default)"),
                    (lr, "lowrisk",  "low-risk borrower (actual: repaid)")]:
    shap.plots.waterfall(sv[i], max_display=13, show=False)
    fig = plt.gcf()
    fig.set_size_inches(9, 6.5)
    fig.suptitle(f"SHAP waterfall — {cap}   (predicted default prob = {proba_samp[i]:.2f})",
                 y=1.02, fontsize=11, fontweight="bold")
    savefig(fig, f"shap_04_waterfall_{tag}.png",
            f"SHAP waterfall chart for a {cap}; predicted probability of default "
            f"{proba_samp[i]:.2f}. Red bars push toward default, blue toward repaid.")


saved -> outputs/shap_04_waterfall_highrisk.png   |  SHAP waterfall chart for a high-risk borrower (actual: default); predicted probability of default 0.81. Red bars push toward default, blue toward repaid.


saved -> outputs/shap_04_waterfall_lowrisk.png   |  SHAP waterfall chart for a low-risk borrower (actual: repaid); predicted probability of default 0.08. Red bars push toward default, blue toward repaid.


## 6. SHAP stability check (proposal risk R4)

Chen et al. (2024) show SHAP rankings can get unstable under class imbalance,
which was one of the risks I flagged in my proposal (R4). To check whether that's
a problem here, I recompute the global importance ranking on five independent
stratified 2,500-row samples (different seeds each time) and look at how
correlated the rankings are across runs.


In [8]:
from scipy.stats import spearmanr

rank_runs = []
for seed in range(5):
    si, _ = train_test_split(np.arange(len(Xte)), train_size=2500,
                             stratify=y_test.values, random_state=100 + seed)
    sv_s = explainer(Xte.iloc[si])
    rank_runs.append(pd.Series(np.abs(sv_s.values).mean(axis=0), index=feat_names))

R = pd.concat(rank_runs, axis=1)
# only look at the features that actually carry any weight - no point checking
# stability on features that are near-zero in every run anyway
top_feats = R.mean(axis=1).sort_values(ascending=False).head(25).index
Rt = R.loc[top_feats]
corr = np.array([[spearmanr(Rt[a], Rt[b]).statistic for b in Rt.columns] for a in Rt.columns])
off = corr[np.triu_indices_from(corr, k=1)]

top10_sets = [set(R[c].sort_values(ascending=False).head(10).index) for c in R.columns]
jaccard = np.mean([len(top10_sets[a] & top10_sets[b]) / len(top10_sets[a] | top10_sets[b])
                   for a in range(5) for b in range(a + 1, 5)])
cv_top = (Rt.std(axis=1) / Rt.mean(axis=1)).mean()   # coefficient of variation of the top-25

print(f"5 independent stratified samples of 2,500 test rows:")
print(f"  Spearman rank correlation of the top-25 importances: "
      f"mean {off.mean():.3f}  (min {off.min():.3f})")
print(f"  Mean Jaccard overlap of the top-10 feature sets      : {jaccard:.3f}")
print(f"  Mean coeff. of variation of top-25 |SHAP| across runs : {cv_top:.3f}")
print("\n-> the SHAP importance ranking is stable across resamples; the leading "
      "drivers do not depend on the particular sample drawn (cf. Chen et al., 2024).")

pd.DataFrame({"feature": R.index, "mean_abs_shap": R.mean(axis=1).values,
              "std_across_runs": R.std(axis=1).values}) \
  .sort_values("mean_abs_shap", ascending=False) \
  .to_csv(os.path.join(OUT_DIR, "shap_stability.csv"), index=False)


5 independent stratified samples of 2,500 test rows:
  Spearman rank correlation of the top-25 importances: mean 0.996  (min 0.994)
  Mean Jaccard overlap of the top-10 feature sets      : 1.000
  Mean coeff. of variation of top-25 |SHAP| across runs : 0.012

-> the SHAP importance ranking is stable across resamples; the leading drivers do not depend on the particular sample drawn (cf. Chen et al., 2024).


## 7. From SHAP to lending guidance (RQ4)

| Behavioural driver (SHAP) | Direction | Practical reading for a lender |
|---|---|---|
| Late instalment ratio ↑ | → default | A borrower who's repeatedly paid late on prior loans is materially higher risk, regardless of their external score. |
| On-time instalment ratio ↑ | → repaid | A strong record of punctual payment is a positive signal even for a thin-file applicant. |
| Recent credit-card utilisation ↑ | → default | High, rising use of revolving credit right before applying points to liquidity stress. |
| Granted-to-requested amount ratio ↓ | → default | Repeatedly being offered less than requested (or refused) on prior applications is a warning sign. |
| Prior-application refusal rate ↑ | → default | A history of refusals with the same lender predicts default on the current loan. |
| Months since last delinquency ↓ | → default | A recent delinquency weighs more heavily than an old one. |
| Number of past instalments ↑ | → repaid | Simply having a longer, observable repayment history lowers predicted risk. |

**What this means in practice:** for MSME-proxy borrowers with limited financial
documentation, the engineered repayment-behaviour and prior-application features
give lenders a defensible, auditable basis for a decision - and the SHAP waterfall
charts provide exactly the kind of per-applicant reason codes that adverse-action
/ fair-lending regulation requires (Bussmann et al., 2021).


## 8. Stage 5 summary

- **RQ1 - most predictive behavioural features** (`shap_03`,
  `outputs/shap_feature_importance.csv`): after the external credit scores, the
  strongest signals are **repayment punctuality** (late-instalment ratio, number
  of past instalments), **recent credit-card utilisation**, and
  **previous-application behaviour** (granted-to-requested ratio, refusal rate).
- **RQ4 - decision-level explanation**: the beeswarm (`shap_02`) shows consistent,
  intuitive effect directions; the waterfall charts (`shap_04_*`) give
  per-applicant reason codes suitable for adverse-action explanations.
- **Stability (proposal R4)**: SHAP importance rankings hold up across five
  independent stratified samples (`outputs/shap_stability.csv`) - high Spearman
  rank correlation and top-10 overlap, so I'm confident this isn't just noise.
- Figures: `shap_01` global importance, `shap_02` beeswarm,
  `shap_03` behavioural importance, `shap_04_waterfall_highrisk/lowrisk`.

That's the whole five-stage pipeline done. All the figures and tables for
Chapters 4–5 are sitting in `outputs/`.
